In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Load the generated dataset
df = pd.read_csv("../data/london_rentals.csv")

# --- 1. DEFINING THE "TRUTH" (From your generation script) ---
INTERCEPT = 7.5
BETA_ROOM = 0.15
BETA_DIST = -0.05
BETA_UNDERGROUND = 0.10
SIGMA = 0.1  # The market noise/volatility
AGENT_PREMIUM = 0.15

# Mappings for categorical logic
TYPE_MAP = {'House': 0.2, 'Flat': 0.0}
OUTDOOR_MAP = {'Garden': 0.15, 'Terrace': 0.1, 'Balcony': 0.05, 'Nothing': 0.0}

# --- 2. FEATURE ENGINEERING ---
# We transform everything into log-space because the math is additive there.
df['log_rent'] = np.log(df['monthly_rent_gbp'])

# Calculate the "Intrinsic Value" of the property (everything except the Agent markup)
# This is our mu (mean) for the 'Owner' hypothesis.
df['expected_log_rent_base'] = (
    INTERCEPT + 
    (df['n_rooms'] * BETA_ROOM) +
    (df['dist_centre_km'] * BETA_DIST) +
    (df['near_underground'] * BETA_UNDERGROUND) +
    df['property_type'].map(TYPE_MAP) +
    df['outdoor_space'].map(OUTDOOR_MAP)
)

print(f"Dataset loaded. Total properties: {len(df)}")
print(f"Reference Sigma: {SIGMA} (approx {SIGMA*100}% price fluctuation)")
df[['monthly_rent_gbp', 'log_rent', 'expected_log_rent_base']].head()

Dataset loaded. Total properties: 1000
Reference Sigma: 0.1 (approx 10.0% price fluctuation)


,monthly_rent_gbp,log_rent,expected_log_rent_base
0,3102.955547,8.040110,7.990439
1,3157.070671,8.057400,8.071226
2,3667.633778,8.207302,8.142533
3,4132.482893,8.326634,8.174331
4,1771.819406,7.479762,7.503178


In [2]:
# --- BAYESIAN LIKELIHOOD CALCULATION ---

"""
LOGIC:
Our generative model defined price as: 
    ln(Price) = expected_log_rent + Noise(0, sigma^2)

To perform inference, we calculate the 'Likelihood'—the probability of observing 
the actual rent given a specific hypothesis (Owner vs. Agent). 

MATH:
If the noise is Gaussian, the likelihood follows the Normal PDF:
    P(Data | Hypothesis) = (1 / (σ * sqrt(2π))) * exp(-0.5 * ((x - μ) / σ)^2)

Where:
    x = The observed 'log_rent'
    μ = Our calculated 'expected_log_rent_base' (plus premium if Agent)
    σ = The 'SIGMA' (0.1) representing market volatility
"""

# Hypothesis A: The property is an Owner listing (No Premium)
# Here, μ = expected_log_rent_base
df['likelihood_owner'] = stats.norm.pdf(
    df['log_rent'], 
    loc=df['expected_log_rent_base'], 
    scale=SIGMA
)

# Hypothesis B: The property is an Agent listing (Includes +0.15 Premium)
# Here, μ = expected_log_rent_base + 0.15
df['likelihood_agent'] = stats.norm.pdf(
    df['log_rent'], 
    loc=df['expected_log_rent_base'] + AGENT_PREMIUM, 
    scale=SIGMA
)

# Preview the results
print("Likelihoods calculated based on Normal PDF of residuals.")
df[['monthly_rent_gbp', 'listing_type', 'likelihood_owner', 'likelihood_agent']].head()

Likelihoods calculated based on Normal PDF of residuals.


,monthly_rent_gbp,listing_type,likelihood_owner,likelihood_agent
0,3102.955547,Owner,3.526423,2.411756
1,3157.070671,Owner,3.951472,1.042573
2,3667.633778,Owner,3.234571,2.774385
3,4132.482893,Owner,1.250867,3.988365
4,1771.819406,Owner,3.881543,0.886923


In [3]:
# --- MODEL COMPARISON: POSTERIOR ODDS RATIO ---

# 1. Calculate the Bayes Factor (Likelihood Ratio)
# How much better does the Agent model explain the price than the Owner model?
df['bayes_factor'] = df['likelihood_agent'] / df['likelihood_owner']

# 2. Prior Odds (0.5 / 0.5 = 1)
prior_odds = 1.0

# 3. Posterior Odds Ratio
df['posterior_odds'] = df['bayes_factor'] * prior_odds

# 4. Decision Logic
# Ratio > 1 -> Evidence favors Agent
# Ratio < 1 -> Evidence favors Owner
print("Odds Ratios calculated. Evidence check:")
df[['monthly_rent_gbp', 'listing_type', 'bayes_factor', 'posterior_odds']].head(10)

Odds Ratios calculated. Evidence check:


,monthly_rent_gbp,listing_type,bayes_factor,posterior_odds
0,3102.955547,Owner,0.683910,0.683910
1,3157.070671,Owner,0.263844,0.263844
2,3667.633778,Owner,0.857729,0.857729
3,4132.482893,Owner,3.188481,3.188481
4,1771.819406,Owner,0.228497,0.228497
5,4523.793073,Owner,0.228503,0.228503
6,3011.560738,Owner,3.468836,3.468836
7,3277.097326,Owner,1.026497,1.026497
8,1665.001388,Owner,0.160540,0.160540
9,3251.304470,Owner,0.732597,0.732597


In [5]:
# --- ACCURACY EVALUATION ---

# 1. The Decision: Pick the model with the higher odds
df['predicted_type'] = np.where(df['posterior_odds'] > 1, 'Agent', 'Owner')

# 2. Check if the prediction matches the ground truth
df['is_correct'] = df['predicted_type'] == df['listing_type']

# 3. Calculate overall accuracy percentage
accuracy = df['is_correct'].mean() * 100

print(f"--- Model Accuracy: {accuracy:.2f}% ---")

# 4. Breakdown by class (Confusion Matrix)
print("\nPrediction Breakdown:")
print(pd.crosstab(df['listing_type'], df['predicted_type']))

--- Model Accuracy: 78.20% ---

Prediction Breakdown:
predicted_type  Agent  Owner
listing_type                
Agent             391    109
Owner             109    391
